# Performance Testing

## ideas
- TSTR (Train-Synthetic-Test-Real)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [4]:
real = pd.read_csv("datasets/transfer.csv", engine="pyarrow")
synth = pd.read_csv("output/output-gpu.csv", engine="pyarrow")

In [5]:
numeric_cols = real.select_dtypes(include=np.number).columns.intersection(
    synth.select_dtypes(include=np.number).columns
)

selected_cols = [
    'st_dirs',
    'st_successful',
    'st_failed',
    'st_expired',
    'st_canceled',
    'st_bytes_xfered',
    'st_faults'
]

# Ensure both datasets contain these columns
real = real[selected_cols].dropna()
synth = synth[selected_cols].dropna()

print(f"Validating {len(selected_cols)} transfer-related features.\n")

Validating 7 transfer-related features.



In [6]:
stats_real = real.describe().T[['mean', 'std', 'min', 'max']]
stats_synth = synth.describe().T[['mean', 'std', 'min', 'max']]

In [7]:
summary = pd.concat(
    [stats_real.add_suffix('_real'), stats_synth.add_suffix('_synth')],
    axis=1
)
print("Descriptive statistics (real vs synthetic):")
print(summary)

Descriptive statistics (real vs synthetic):
                    mean_real      std_real  min_real      max_real  \
st_dirs          3.104170e+02  3.320389e+04       0.0  8.791225e+07   
st_successful    4.771814e+03  2.615866e+05       0.0  5.114859e+08   
st_failed        4.514573e+01  4.917308e+04       0.0  1.440816e+08   
st_expired       5.376443e+02  1.868197e+05       0.0  4.457491e+08   
st_canceled      9.817168e+02  1.723540e+05       0.0  2.807579e+08   
st_bytes_xfered  5.569412e+10  2.633040e+12       0.0  7.802815e+15   
st_faults        6.310797e+00  1.544917e+02       0.0  4.429300e+05   

                   mean_synth     std_synth     min_synth     max_synth  
st_dirs          6.559936e-01  1.921749e+01  0.000000e+00  5.866300e+04  
st_successful    5.842361e+06  1.792593e+10  0.000000e+00  6.931934e+13  
st_failed        1.298519e-04  1.690711e-02  0.000000e+00  6.400000e+01  
st_expired       9.916900e-03  2.109334e-01  0.000000e+00  2.560000e+02  
st_canceled      

In [8]:
synth.loc[synth["st_bytes_xfered"] > real["st_bytes_xfered"].max(), "st_bytes_xfered"] = real["st_bytes_xfered"].max()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
 
# Plot the distributions
col = "st_bytes_xfered"
plt.figure(figsize=(8, 5))

sns.kdeplot(real[col], label="Real", fill=True, alpha=0.5)

sns.kdeplot(synth[col], label="Synth", fill=True, alpha=0.5)
plt.xscale("log")
 
plt.title("Distribution of col(Real vs Synth)")

plt.xlabel(col)

plt.ylabel("Density")

plt.legend()

plt.show()

 

In [9]:
ks_results, wd_results = {}, {}
for col in selected_cols:
    ks = ks_2samp(real[col], synth[col]).statistic
    wd = wasserstein_distance(real[col], synth[col])
    ks_results[col] = ks
    wd_results[col] = wd

ks_df = pd.DataFrame({'KS_stat': ks_results, 'Wasserstein': wd_results})
print("\nDistribution similarity (lower is better):")
print(ks_df.sort_values('KS_stat'))


Distribution similarity (lower is better):
                  KS_stat   Wasserstein
st_failed        0.000128  4.514560e+01
st_expired       0.004195  5.376343e+02
st_canceled      0.010252  9.816639e+02
st_faults        0.025880  6.185338e+00
st_dirs          0.059575  3.097610e+02
st_bytes_xfered  0.064457  9.014562e+11
st_successful    0.067371  5.841106e+06


In [10]:
print("real")
print(np.percentile(real.st_bytes_xfered, 99))
print(np.percentile(real.st_bytes_xfered, 95))
print(np.percentile(real.st_bytes_xfered, 90))
print("synth")
print(np.percentile(synth.st_bytes_xfered, 99))
print(np.percentile(synth.st_bytes_xfered, 95))
print(np.percentile(synth.st_bytes_xfered, 90))

# clipping or re-drawing

real
566944388258.2393
36888780651.14996
6280405094.700006
synth
566944355070.0
36888780651.0
6280405095.0


In [11]:
corr_real = real.corr()
corr_synth = synth.corr()

# Drop columns that annihilate correlation
zero_var = real.columns[(real.nunique() <= 1) | (synth.nunique() <= 1)]
real_corr_input = real.drop(columns=zero_var)
synth_corr_input = synth.drop(columns=zero_var)
corr_real = real_corr_input.corr()
corr_synth = synth_corr_input.corr()
corr_diff = np.linalg.norm(corr_real - corr_synth, 'fro')

corr_diff_matrix = (corr_real - corr_synth).abs()

# Flatten to pairs for readability
corr_diff_pairs = (
    corr_diff_matrix
    .where(np.triu(np.ones(corr_diff_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_diff_pairs.columns = ['Feature1', 'Feature2', 'Abs_Diff']

# Sort by difference magnitude
corr_diff_pairs = corr_diff_pairs.sort_values('Abs_Diff', ascending=False)
print("Top correlation differences:")
print(corr_diff)
print(corr_diff_pairs.head(10))


Top correlation differences:
0.7496369889442691
           Feature1         Feature2  Abs_Diff
2           st_dirs       st_expired  0.330682
0           st_dirs    st_successful  0.319993
3           st_dirs      st_canceled  0.205483
9     st_successful  st_bytes_xfered  0.095165
8     st_successful      st_canceled  0.084206
17       st_expired        st_faults  0.060778
4           st_dirs  st_bytes_xfered  0.056129
7     st_successful       st_expired  0.040280
6     st_successful        st_failed  0.027704
20  st_bytes_xfered        st_faults  0.020531
